In [1]:
# Cell 1 — Setup
import sys, json
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import HTML, display as _display
import html as _html

here = Path().resolve()
REPO_ROOT = here
for _ in range(4):
    if (REPO_ROOT / 'grammar_parser').is_dir():
        break
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import spacy
from grammar_parser import Group1Parser, Group2Parser, Group3Parser, Group4Parser

nlp     = spacy.load('en_core_web_sm')
parser1 = Group1Parser(nlp, resolve=True)
parser2 = Group2Parser(nlp, resolve=True)
parser3 = Group3Parser(nlp, resolve=True)
parser4 = Group4Parser(nlp, resolve=True)
parsers = [parser1, parser2, parser3, parser4]

DIMENSION_MAP = {
    "CLAUSES":"A","QUESTIONS":"A","REPORTED SPEECH":"A",
    "CONJUNCTIONS":"A","DISCOURSE MARKERS":"A",
    "PAST":"B","PRESENT":"B","FUTURE":"B","PASSIVES":"B","VERBS":"B","NEGATION":"B",
    "PRONOUNS":"C","DETERMINERS":"C","NOUNS":"C",
    "ADJECTIVES":"C","ADVERBS":"C","PREPOSITIONS":"C",
    "MODALITY":"D","FOCUS":"D",
}
DIMENSION_LABELS = {
    "A": "Sentence Architecture", "B": "Tense & Aspect",
    "C": "Nominal Precision",     "D": "Modal & Function",
}
DIM_BG   = {"A":"#DBEAFE","B":"#FEF3C7","C":"#D1FAE5","D":"#EDE9FE","?":"#F3F4F6"}
DIM_TEXT = {"A":"#1D4ED8","B":"#92400E","C":"#065F46","D":"#5B21B6","?":"#374151"}
DIM_MARK = {"A":"#93C5FD","B":"#FCD34D","C":"#6EE7B7","D":"#C4B5FD","?":"#D1D5DB"}
CEFR_BG  = {"A1":"#D1FAE5","A2":"#6EE7B7","B1":"#FEF3C7",
            "B2":"#FDE68A","C1":"#FECACA","C2":"#FCA5A5"}
CEFR_TXT = {"A1":"#065F46","A2":"#064E3B","B1":"#78350F",
            "B2":"#92400E","C1":"#991B1B","C2":"#7F1D1D"}

# ── Richness score weights ────────────────────────────────────────────────────
# Score = 60 * level_score + 40 * variety_score  (0–100)
LEVEL_WEIGHT   = 0.60   # avg CEFR level of assigned structures (A2=0 .. C2=1)
VARIETY_WEIGHT = 0.40   # distinct grammar categories / VARIETY_CEILING

# 15 out of 18 possible categories needed for full variety score.
# A typical rich paragraph reaches 8–10, so this is deliberately hard to max out.
VARIETY_CEILING = 15

LEVEL_MIN, LEVEL_MAX = 2, 6   # A2=0, C2=1  (A1 removed from ruleset)

_HIGHLIGHT_CSS = """<style>
.m-hl     { font-weight:700; border-radius:3px; padding:1px 3px; }
.m-hl.m-A { background:#93C5FD; color:#1D4ED8; }
.m-hl.m-B { background:#FCD34D; color:#92400E; }
.m-hl.m-C { background:#6EE7B7; color:#065F46; }
.m-hl.m-D { background:#C4B5FD; color:#5B21B6; }
.m-hl.m-x { background:#D1D5DB; color:#374151; }
.c-hl     { border-radius:3px; padding:1px 2px;
            background:transparent; transition:background 0.15s; }
.para-block:hover .c-hl.c-A { background:#DBEAFE; color:#1D4ED8; }
.para-block:hover .c-hl.c-B { background:#FEF3C7; color:#92400E; }
.para-block:hover .c-hl.c-C { background:#D1FAE5; color:#065F46; }
.para-block:hover .c-hl.c-D { background:#EDE9FE; color:#5B21B6; }
.para-block:hover .c-hl.c-x { background:#F3F4F6; color:#374151; }
</style>"""

print(f"Repo root : {REPO_ROOT}")
print(f"spaCy     : {nlp.meta['name']} v{nlp.meta['version']}")
print(f"Parsers   : {sum(len(p.structures) for p in parsers)} structures total")

Repo root : C:\Users\marta\Documents\PROJECT BARCELONA\GRAMMAR
spaCy     : core_web_sm v3.8.0
Parsers   : 133 structures total


In [2]:
# Cell 2 — Load paragraph sessions
PARTITIONS = [
    REPO_ROOT / 'processed_data' / 'sentences' / 'Student-2' / 'lesson-1' / 'paragraphs_custom.json',
    REPO_ROOT / 'processed_data' / 'sentences' / 'Student-1' / 'lesson-3' / 'paragraphs_custom.json',
]

sessions = []
for p in PARTITIONS:
    data = json.loads(p.read_text(encoding='utf-8'))
    sessions.append(data)
    n_sents = sum(len(para['sentences']) for para in data['paragraphs'])
    print(f"  {data['student']} / {data['lesson']} — "
          f"{len(data['paragraphs'])} paragraphs, {n_sents} sentences")


  Student-2 / lesson-1 — 10 paragraphs, 368 sentences
  Student-1 / lesson-3 — 10 paragraphs, 303 sentences


In [3]:
# Cell 3 — Run grammar parsers per paragraph
def run_pipeline_paragraphs(session, nlp, parsers):
    """Returns {paragraph_id: [match_dict, ...]} with char offsets attached."""
    results = {}
    for para in session['paragraphs']:
        pid       = para['id']
        sentences = [s['text'] for s in para['sentences']]
        docs      = list(nlp.pipe(sentences))
        matches   = []
        for sent_text, doc in zip(sentences, docs):
            for parser in parsers:
                for m in parser.parse(doc):
                    span = doc[m['start_token']:m['end_token']]
                    ctx  = doc[m['context_start_token']:m['context_end_token']]
                    m = dict(m)
                    m['sentence_text']      = sent_text
                    m['start_char']         = span.start_char
                    m['end_char']           = span.end_char
                    m['context_start_char'] = ctx.start_char
                    m['context_end_char']   = ctx.end_char
                    matches.append(m)
        results[pid] = matches
    return results

all_results = {}
for session in sessions:
    key = f"{session['student']} / {session['lesson']}"
    all_results[key] = run_pipeline_paragraphs(session, nlp, parsers)
    total = sum(len(v) for v in all_results[key].values())
    print(f"  {key} — {total} grammar events across {len(all_results[key])} paragraphs")


  Student-2 / lesson-1 — 2540 grammar events across 10 paragraphs


  Student-1 / lesson-3 — 2680 grammar events across 10 paragraphs


In [4]:
# Cell 4 — Richness metric computation
#
# Score = 60 * level_score + 40 * variety_score  (0–100)
#   level    : (avg assigned CEFR numeric - 2) / 4   (A2=0, C2=1)
#   variety  : distinct grammar categories / VARIETY_CEILING (15)
#
# Dimension coverage (A/B/C/D) is shown informatively but not in the score.

def _assigned_groups(matches):
    """One entry per (span, category, sentence), keeping highest-level match."""
    groups = {}
    for m in matches:
        key = (m['start_token'], m['end_token'], m['category'], m['sentence_text'])
        if key not in groups or m['lowest_level_numeric'] > groups[key]['lowest_level_numeric']:
            groups[key] = m
    return list(groups.values())


def compute_richness(matches, n_sentences):
    assigned = _assigned_groups(matches)

    if not assigned:
        return {
            'score': 0, 'label': 'Sparse', 'color': '#dc2626',
            'variety': 0.0, 'level': 0.0, 'density': 0.0,
            'distinct_categories': [], 'dims_present': [],
            'level_distribution': {}, 'top_match': None,
            'avg_level_str': '-', 'n_assigned': 0,
        }

    # Sophistication: average assigned CEFR level (60% of score)
    level_nums  = [m['lowest_level_numeric'] for m in assigned]
    avg_level   = sum(level_nums) / len(level_nums)
    level_score = max(0.0, min(1.0, (avg_level - LEVEL_MIN) / (LEVEL_MAX - LEVEL_MIN)))

    # Variety: distinct grammar categories out of 18 possible (40% of score)
    # Ceiling = 15: needs 15/18 categories for full points — hard to max out
    distinct_cats = sorted({m['category'] for m in assigned})
    variety_score = min(len(distinct_cats) / VARIETY_CEILING, 1.0)

    # Dimension coverage (informative only — not in score)
    dims_present = sorted({DIMENSION_MAP.get(m['category'], '?') for m in assigned} - {'?'})

    # Density: grammar events per sentence (auxiliary)
    density = len(assigned) / n_sentences if n_sentences > 0 else 0.0

    # Composite score
    raw   = LEVEL_WEIGHT * level_score + VARIETY_WEIGHT * variety_score
    score = max(0, min(100, round(raw * 100)))

    if   score >= 70: label, color = 'Rich',     '#15803d'
    elif score >= 50: label, color = 'Moderate', '#d97706'
    elif score >= 30: label, color = 'Basic',    '#ea580c'
    else:             label, color = 'Sparse',   '#dc2626'

    lv_labels = ['A1','A2','B1','B2','C1','C2']
    avg_str   = lv_labels[max(0, min(5, round(avg_level) - 1))]

    return {
        'score':               score,
        'label':               label,
        'color':               color,
        'variety':             variety_score,
        'level':               level_score,
        'density':             density,
        'distinct_categories': distinct_cats,
        'dims_present':        dims_present,
        'level_distribution':  dict(Counter(m['lowest_level'] for m in assigned)),
        'top_match':           max(assigned, key=lambda m: m['lowest_level_numeric']),
        'avg_level_str':       avg_str,
        'n_assigned':          len(assigned),
    }

print('compute_richness defined.')

compute_richness defined.


In [5]:
# Cell 5 — Per-paragraph HTML visualisation
# Each paragraph card: score badge, component breakdown bars (Level 60 + Variety 40),
# dimension pills (informative), best-shot structure, CEFR distribution,
# and sentences with two-level highlighting.

def _dim_pill(dim):
    bg, fg = DIM_BG.get(dim,'#F3F4F6'), DIM_TEXT.get(dim,'#374151')
    lbl    = DIMENSION_LABELS.get(dim, dim)
    return (f'<span style="background:{bg};color:{fg};padding:2px 7px;'
            f'border-radius:10px;font-size:11px;font-weight:600;margin-right:4px;">'
            f'{dim} · {lbl}</span>')


def _cefr_dist_bar(level_dist, n_assigned):
    parts = []
    for lv in ['A2','B1','B2','C1','C2']:
        n = level_dist.get(lv, 0)
        if not n:
            continue
        bg = CEFR_BG.get(lv,'#E5E7EB')
        fg = CEFR_TXT.get(lv,'#374151')
        parts.append(
            f'<span style="display:inline-flex;align-items:center;gap:3px;'
            f'background:{bg};color:{fg};padding:1px 6px;border-radius:6px;'
            f'font-size:11px;font-weight:700;margin-right:4px;">'
            f'{lv} <span style="opacity:0.7;font-weight:400;">{n}</span></span>'
        )
    return ''.join(parts)


def _score_badge(score, label, color):
    return (f'<span style="background:{color};color:#fff;padding:3px 10px;'
            f'border-radius:8px;font-size:13px;font-weight:700;">'
            f'{score} · {label}</span>')


def _comp_bars(rich):
    def _bar(label, value, max_pts, color):
        return (
            f'<div style="display:flex;gap:4px;align-items:center;margin:3px 0;">'
            f'<span style="font-size:10px;color:#9CA3AF;min-width:56px;">{label}</span>'
            f'<div style="background:#E5E7EB;border-radius:4px;height:7px;flex:1;">'
            f'<div style="background:{color};width:{value*100:.0f}%;height:7px;border-radius:4px;"></div></div>'
            f'<span style="font-size:10px;color:#6B7280;min-width:32px;text-align:right;">'
            f'{value*max_pts:.0f}/{max_pts}</span>'
            f'</div>'
        )
    return (
        _bar('Level',   rich['level'],   60, '#FCD34D') +
        _bar('Variety', rich['variety'], 40, '#93C5FD')
    )


def _highlight_sentence(text, match_spans, context_spans):
    n    = len(text)
    tier = [None] * n
    for s, e, dim in context_spans:
        css = dim if dim in 'ABCD' else 'x'
        for i in range(max(0,s), min(e,n)):
            if tier[i] is None:
                tier[i] = ('ctx', css)
    for s, e, dim in match_spans:
        css = dim if dim in 'ABCD' else 'x'
        for i in range(max(0,s), min(e,n)):
            tier[i] = ('match', css)
    parts, i = [], 0
    while i < n:
        t = tier[i]
        if t is None:
            j = i
            while j < n and tier[j] is None: j += 1
            parts.append(_html.escape(text[i:j])); i = j
        else:
            kind, css = t
            j = i
            while j < n and tier[j] == t: j += 1
            chunk = _html.escape(text[i:j])
            cls   = f'm-hl m-{css}' if kind == 'match' else f'c-hl c-{css}'
            tag   = 'mark' if kind == 'match' else 'span'
            parts.append(f'<{tag} class="{cls}">{chunk}</{tag}>')
            i = j
    return ''.join(parts)


def _render_sentences(para_matches):
    by_sent = defaultdict(list)
    for m in para_matches:
        by_sent[m['sentence_text']].append(m)
    parts = []
    for sent_text, sent_matches in by_sent.items():
        match_spans, ctx_spans = {}, {}
        for m in sent_matches:
            dim = DIMENSION_MAP.get(m['category'], '?')
            mk  = (m['start_char'], m['end_char'])
            ck  = (m['context_start_char'], m['context_end_char'])
            if mk not in match_spans: match_spans[mk] = dim
            if ck not in ctx_spans:   ctx_spans[ck]   = dim
        hl = _highlight_sentence(
            sent_text,
            [(s,e,d) for (s,e),d in match_spans.items()],
            [(s,e,d) for (s,e),d in ctx_spans.items()],
        )
        parts.append(f'<div style="font-size:13px;line-height:1.7;padding:3px 0;">{hl}</div>')
    return '\n'.join(parts)


def display_richness_report(sessions, all_results):
    blocks = [_HIGHLIGHT_CSS]

    for session in sessions:
        key   = f"{session['student']} / {session['lesson']}"
        res   = all_results[key]
        cards = []

        for para in session['paragraphs']:
            pid       = para['id']
            sentences = [s['text'] for s in para['sentences']]
            matches   = res.get(pid, [])
            rich      = compute_richness(matches, len(sentences))

            # Dimension pills (informative — not in score)
            dim_pills = ''.join(_dim_pill(d) for d in rich['dims_present']) or \
                        '<span style="color:#9CA3AF;font-size:11px;">none detected</span>'

            # Best-shot structure
            top = rich['top_match']
            top_html = ''
            if top:
                tdim = DIMENSION_MAP.get(top['category'], '?')
                tbg  = CEFR_BG.get(top['lowest_level'], '#F3F4F6')
                tfg  = CEFR_TXT.get(top['lowest_level'], '#374151')
                top_html = (
                    f'<div style="margin:8px 0;padding:8px 10px;background:#F9FAFB;'
                    f'border-radius:6px;border-left:3px solid {DIM_MARK.get(tdim,"#D1D5DB")};">'
                    f'<span style="font-size:10px;color:#9CA3AF;text-transform:uppercase;'
                    f'letter-spacing:.05em;">Best shot</span><br>'
                    f'<span style="font-size:12px;font-weight:600;color:#111827;">'
                    f'{_html.escape(top["guideword"])}</span>&nbsp;'
                    f'<span style="background:{tbg};color:{tfg};padding:1px 6px;'
                    f'border-radius:6px;font-size:11px;font-weight:700;">{top["lowest_level"]}</span>'
                    f'<br><span style="font-size:11px;color:#6B7280;font-style:italic;">'
                    f'{_html.escape(top.get("explanation",""))}</span>'
                    f'</div>'
                )

            sent_html = _render_sentences(matches) if matches else \
                '<span style="color:#9CA3AF;font-size:12px;font-style:italic;">No structures detected.</span>'

            cards.append(f'''
            <div class="para-block" style="margin:10px 0;padding:14px 16px;
                        border:1px solid #E5E7EB;border-radius:8px;
                        font-family:system-ui,-apple-system,sans-serif;background:#FAFAFA;">
              <div style="display:flex;justify-content:space-between;
                          align-items:flex-start;margin-bottom:8px;">
                <div>
                  <span style="font-size:11px;color:#9CA3AF;letter-spacing:.04em;">
                    P{pid:02d}&nbsp;·&nbsp;{len(sentences)} sent&nbsp;·&nbsp;{rich["n_assigned"]} structs
                    &nbsp;·&nbsp;{rich["density"]:.1f}/sent&nbsp;·&nbsp;avg {rich["avg_level_str"]}
                  </span><br>
                  <span style="font-size:14px;font-weight:600;color:#1F2937;">
                    {_html.escape(para["label"])}
                  </span>
                </div>
                <div style="text-align:right;">
                  {_score_badge(rich["score"], rich["label"], rich["color"])}
                  <div style="margin-top:6px;">{_cefr_dist_bar(rich["level_distribution"], max(rich["n_assigned"],1))}</div>
                </div>
              </div>
              <div style="margin:6px 0 8px;">{dim_pills}</div>
              {_comp_bars(rich)}
              {top_html}
              <div style="margin-top:10px;padding:8px 10px;background:#fff;
                          border-radius:6px;border-left:3px solid #D1D5DB;">
                {sent_html}
              </div>
            </div>''')

        blocks.append(f'''
        <div style="margin:24px 0;font-family:system-ui,-apple-system,sans-serif;">
          <h3 style="color:#1F2937;border-bottom:2px solid #E5E7EB;padding-bottom:6px;margin-bottom:0;">
            {_html.escape(key)}
          </h3>
          {"".join(cards)}
        </div>''')

    _display(HTML('\n'.join(blocks)))


display_richness_report(sessions, all_results)

In [6]:
# Cell 6 — Summary table (all sessions)

def display_summary_table(sessions, all_results):
    def _badge(score, label, color):
        return (f'<span style="background:{color};color:#fff;padding:1px 8px;'
                f'border-radius:6px;font-size:11px;font-weight:700;">{score}</span> '
                f'<span style="font-size:11px;color:{color};font-weight:600;">{label}</span>')

    rows = []
    for session in sessions:
        key = f"{session['student']} / {session['lesson']}"
        res = all_results[key]
        rows.append(
            f'<tr><td colspan="9" style="padding:10px 8px 4px;font-weight:700;'
            f'font-size:13px;color:#1F2937;border-top:2px solid #E5E7EB;">'
            f'{_html.escape(key)}</td></tr>'
        )
        for para in session['paragraphs']:
            pid       = para['id']
            sentences = [s['text'] for s in para['sentences']]
            matches   = res.get(pid, [])
            rich      = compute_richness(matches, len(sentences))
            n_words   = sum(len(s.split()) for s in sentences)
            dims_str  = ' '.join(rich['dims_present']) or '—'

            lvl_cells = ''.join(
                f'<span style="background:{CEFR_BG.get(lv,"#F3F4F6")};'
                f'color:{CEFR_TXT.get(lv,"#374151")};padding:0 5px;'
                f'border-radius:4px;font-size:10px;font-weight:700;margin-right:2px;">'
                f'{rich["level_distribution"].get(lv,0)}</span>'
                for lv in ['A2','B1','B2','C1','C2']
            )
            rows.append(
                f'<tr style="border-bottom:1px solid #F3F4F6;">'
                f'<td style="padding:5px 8px;font-weight:600;">P{pid:02d}</td>'
                f'<td style="padding:5px 8px;font-size:12px;max-width:180px;">'
                f'{_html.escape(para["label"][:42])}{"…" if len(para["label"])>42 else ""}</td>'
                f'<td style="padding:5px 8px;text-align:center;">{len(sentences)}</td>'
                f'<td style="padding:5px 8px;text-align:center;">{n_words}</td>'
                f'<td style="padding:5px 8px;text-align:center;">{rich["n_assigned"]}</td>'
                f'<td style="padding:5px 8px;text-align:center;">{len(rich["distinct_categories"])}</td>'
                f'<td style="padding:5px 8px;">{lvl_cells}</td>'
                f'<td style="padding:5px 8px;text-align:center;font-weight:600;">{dims_str}</td>'
                f'<td style="padding:5px 8px;">{_badge(rich["score"],rich["label"],rich["color"])}</td>'
                f'</tr>'
            )

    _display(HTML(f'''
    <div style="font-family:system-ui,-apple-system,sans-serif;">
      <h3 style="color:#1F2937;margin:0 0 12px 0;">Grammar Richness — Summary</h3>
      <table style="width:100%;border-collapse:collapse;font-size:12px;">
        <thead>
          <tr style="background:#F9FAFB;border-bottom:2px solid #E5E7EB;">
            <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;">P#</th>
            <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;">Topic</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Sents</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Words</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Structs</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Categ.</th>
            <th style="padding:6px 8px;color:#6B7280;font-size:10px;text-transform:uppercase;">CEFR dist.</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Dims</th>
            <th style="padding:6px 8px;color:#6B7280;font-size:10px;text-transform:uppercase;">Score</th>
          </tr>
        </thead>
        <tbody>{"".join(rows)}</tbody>
      </table>
    </div>'''))


display_summary_table(sessions, all_results)
